# C3 · extracción óptima — notebook de análisis (`debug`)

**Objeto:** ROXs42Bb  |  **Run:** `ROXs42Bb_realigned`  |  **Spec:** [`docs/spec_C3_codex_optimal_extraction.md`](../../../docs/spec_C3_codex_optimal_extraction.md)

Rehace C3 **dentro del notebook**, con el código a la vista y editable, para probar y ajustar sin tocar `musepipe`. El notebook de auditoría es [`../C3_optimal.ipynb`](../C3_optimal.ipynb).

**C3 produce dos métodos, no uno.** El estimador es el mismo — Horne (1986): por canal, cada píxel pesa por el perfil de PSF esperado y por la inversa de su varianza, `f = Σ M·P·D/V ÷ Σ M·P²/V` — y lo que cambia es **el cubo del que se extrae**:

| variante | cubo | por qué existe |
|---|---|---|
| `optimal_ls` | residual de superficie local (04b) | **mismo fondo que C2**, así que compararlos aísla la ganancia del ponderado óptimo |
| `optimal_psfsub` | cubo de B2 menos el **modelo de PSF de la primaria**, ajustado aquí canal a canal | anticipa el fondo de C4; `ls` vs `psfsub` es el diagnóstico del modelo de halo que consume D1 |

Aquí se hacen **las dos**, en paralelo, y la comparación final las contrasta por separado contra sus productos de la cadena.


In [ ]:
import json, sys
from pathlib import Path

import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

_here = Path.cwd()
ROOT = next(p for p in (_here, *_here.parents) if (p / 'musepipe').is_dir())
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'notebooks'))
import _nbcommon as nb

RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
RD = nb.run_dir(RUN_ID); SD = RD / 'stages'
CFG = json.loads((RD / 'config' / 'config.json').read_text(encoding='utf-8'))['config']
TARGET = nb.run_target(RUN_ID) or nb.display_name(RUN_ID)
print('objeto :', TARGET, '·', nb.display_name(RUN_ID))
print('run    :', RUN_ID)


## 1 · Perillas

Salen del **config resuelto de la etapa**, no del `config.json` crudo: C3 rellena defaults que no están escritos en el run (`x02_local_bkg_annulus_px` hereda de `x01_annulus_bkg_px`, el radio de ajuste de la primaria de `psf_norm_radius_px`…), y copiarlos a mano es exactamente cómo se consigue un notebook que no reproduce la cadena. Cambia lo que quieras **debajo** de la lectura y vuelve a ejecutar.

`WINDOW_RADIUS_PX` es la que más mueve el resultado: define hasta dónde llega el ponderado, y la fracción de PSF que queda fuera la recupera después `apcorr`.


In [ ]:
from musepipe.stages.stage_x02_optimal import stage_x02_config_from_run

X02 = stage_x02_config_from_run(RUN_ID)   # config del run + defaults de la etapa
WINDOW_RADIUS_PX     = float(X02.get('x02_window_radius_px', 8.0))
CLIP_SIGMA           = float(X02.get('x02_clip_sigma', 4.0))
CLIP_MAX_ITER        = int(X02.get('x02_clip_max_iter', 2))
APCORR_MODE          = X02.get('x02_aperture_correction', 'auto')
ERROR_MODE           = X02.get('x02_error_mode', 'auto')
N_CONTROLS           = int(X02.get('x02_control_apertures', 8))
EXCLUDE_ANGLE_DEG    = float(X02.get('x02_control_exclude_angle_deg', 25.0))
LOCAL_BKG_ANNULUS_PX = X02.get('x02_local_bkg_annulus_px')
PRIMARY_FIT_RADIUS   = float(X02.get('x02_primary_fit_radius_px', 25.0))
PRIMARY_EXCL_RADIUS  = float(X02.get('x02_primary_exclude_radius_px', WINDOW_RADIUS_PX))
BAD_WINDOWS_A        = X02.get('x02_bad_windows_A', [])
SKYLINE_WINDOWS_A    = X02.get('x02_skyline_windows_A', [])
INTERPOLATED_WIN_A   = X02.get('x02_interpolated_windows_A', [])

# ---- a partir de aquí, cambia lo que quieras probar ----

for _k, _v in sorted({'ventana (px)': WINDOW_RADIUS_PX, 'clip σ': CLIP_SIGMA,
                      'clip iter': CLIP_MAX_ITER, 'apcorr': APCORR_MODE,
                      'modo error': ERROR_MODE, 'controles': N_CONTROLS,
                      'anillo fondo': LOCAL_BKG_ANNULUS_PX,
                      'radio ajuste primaria': PRIMARY_FIT_RADIUS,
                      'radio exclusión compañero': PRIMARY_EXCL_RADIUS}.items()):
    print(f'  {_k:26s} {_v}')


## 2 · Entradas — **los dos cubos**

`ls` sale del residual de 04b; `psfsub` del cubo de B2. Los dos deben tener la misma forma: la cadena lo exige y para aquí si no (serían dos rejillas distintas).


In [ ]:
qc_b3 = json.loads((SD / 'stage01c_qc.json').read_text(encoding='utf-8'))
OBJECT_YX = tuple(float(v) for v in qc_b3['companion']['pos_yx'])
STAR_YX   = tuple(float(v) for v in qc_b3['primary']['pos_yx'])
PSF_MODEL = json.loads((SD / 'psf_model.json').read_text(encoding='utf-8'))

with fits.open(SD / 'stage02_xcorr_cube_stack.fits') as h:
    STAGE02 = np.asarray(h['CUBES'].data, dtype=float)
    WAVE = np.asarray(h['WAVELENGTH'].data, dtype=float)
    STAT_CUBE = np.asarray(h['STAT'].data, dtype=float) if 'STAT' in h else None
if STAGE02.ndim == 4:
    STAGE02 = STAGE02[0]
if STAT_CUBE is not None and STAT_CUBE.ndim == 4:
    STAT_CUBE = STAT_CUBE[0]
LS_CUBE = np.asarray(fits.getdata(SD / 'cube_residual_local_object.fits'), dtype=float)
assert LS_CUBE.shape == STAGE02.shape, (LS_CUBE.shape, STAGE02.shape)

qc00 = json.loads((SD / 'stage00q_qc.json').read_text(encoding='utf-8'))
qc01 = json.loads((SD / 'stage01_qc.json').read_text(encoding='utf-8'))
m5 = qc00.get('m5_stat', {})
# El STAT crudo se multiplica por el factor POR SPAXEL de M5 (aquí no es 1:
# el DRS subestima la varianza) antes de entrar como peso del estimador.
STAT_FACTOR = float(X02.get('x02_stat_factor_spaxel',
                            m5.get('factor_spaxel_median', 1.0)) or 1.0)
COV_FACTOR  = float(X02.get('x02_covariance_factor_box3',
                            qc01.get('stat', {}).get('covariance_factor_box3', 1.0)) or 1.0)
STAT_STATUS = str(X02.get('x02_stat_status', m5.get('status', 'unknown')))
print('cubos    :', STAGE02.shape, '| STAT:', 'sí' if STAT_CUBE is not None else 'no')
print('compañero:', [round(v, 2) for v in OBJECT_YX], ' primaria:', [round(v, 2) for v in STAR_YX])
print(f'STAT     : factor={STAT_FACTOR:.3f} covarianza={COV_FACTOR:.3f} estado={STAT_STATUS}')


## 3 · Las funciones numéricas, copiadas de `musepipe`

Copia **literal**; edítalas y el resultado cambia. Se importan solo `evaluate_psf_model` (es de C1) y `run_channel_chunks` (paralelismo, no física).

- `finite_values` — de `musepipe/stats.py`
- `robust_sigma` — de `musepipe/stats.py`
- `robust_sigma_axis0` — de `musepipe/stats.py`
- `angular_separation_deg` — de `musepipe/apertures.py`
- `aperture_weights` — de `musepipe/apertures.py`
- `same_radius_control_positions` — de `musepipe/apertures.py`
- `_as_cube` — de `musepipe/extraction/aperture.py`
- `annulus_background_spectrum` — de `musepipe/extraction/aperture.py`
- `_flag_window` — de `musepipe/extraction/aperture.py`
- `channel_flags` — de `musepipe/extraction/aperture.py`
- `aperture_correction_from_psf` — de `musepipe/extraction/aperture.py`
- `circular_window_indices` — de `musepipe/extraction/optimal.py`
- `normalized_psf_window` — de `musepipe/extraction/optimal.py`
- `covariance_factor_for_npix` — de `musepipe/extraction/optimal.py`
- `_channel_estimate` — de `musepipe/extraction/optimal.py`
- `estimate_variance_cube` — de `musepipe/extraction/optimal.py`
- `optimal_raw_spectrum` — de `musepipe/extraction/optimal.py`
- `control_optimal_spectra` — de `musepipe/extraction/optimal.py`
- `psf_image` — de `musepipe/extraction/optimal.py`
- `fit_primary_psf_model_cube` — de `musepipe/extraction/optimal.py`


In [ ]:
# ------------------------------------------------------------------
# COPIA EDITABLE. Fuente: musepipe (ver el chequeo de deriva abajo).
# ------------------------------------------------------------------
from typing import Sequence
import math
import numpy as np
import warnings
from musepipe.psf import evaluate_psf_model      # de C1
from musepipe.parallel import run_channel_chunks  # paralelismo, no física

FLAG_BAD_WINDOW = 1
FLAG_SKYLINE = 2
FLAG_INTERPOLATED = 4
FLAG_CLIPPED = 8


def finite_values(values) -> np.ndarray:
    """Return finite values as a float64 1D array."""

    arr = np.asarray(values, dtype=np.float64)
    return arr[np.isfinite(arr)]


def robust_sigma(values) -> float:
    """Robust 1D sigma estimate using MAD with std fallback."""

    vals = finite_values(values)
    if vals.size == 0:
        return np.nan
    med = np.nanmedian(vals)
    mad = np.nanmedian(np.abs(vals - med))
    sigma = 1.4826 * mad
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = np.nanstd(vals)
    return float(sigma)


def robust_sigma_axis0(values) -> np.ndarray:
    """Robust sigma along axis 0 using MAD with std fallback per column."""

    arr = np.asarray(values, dtype=np.float64)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        med = np.nanmedian(arr, axis=0)
        mad = np.nanmedian(np.abs(arr - med[None, :]), axis=0)
        sigma = 1.4826 * mad
        std = np.nanstd(arr, axis=0)
    bad = ~np.isfinite(sigma) | (sigma <= 0)
    sigma[bad] = std[bad]
    return sigma


def angular_separation_deg(a, b) -> float:
    """Smallest angular separation between two angles in radians, in degrees."""

    return abs(math.degrees(math.atan2(math.sin(a - b), math.cos(a - b))))


def aperture_weights(ny, nx, center_yx, aperture) -> np.ndarray:
    """Build a 2D aperture-weight image."""

    y0, x0 = map(float, center_yx)
    yy, xx = np.mgrid[:ny, :nx]
    rr2 = (yy - y0) ** 2 + (xx - x0) ** 2
    weights = np.zeros((ny, nx), dtype=np.float64)
    kind = aperture["kind"]

    if kind == "pixel":
        y = int(round(y0))
        x = int(round(x0))
        if 0 <= y < ny and 0 <= x < nx:
            weights[y, x] = 1.0
    elif kind == "box":
        size = int(aperture.get("size", 3))
        half = size // 2
        y = int(round(y0))
        x = int(round(x0))
        y1 = max(0, y - half)
        y2 = min(ny, y + half + 1)
        x1 = max(0, x - half)
        x2 = min(nx, x + half + 1)
        weights[y1:y2, x1:x2] = 1.0
    elif kind == "circle":
        radius = float(aperture["radius_px"])
        weights[rr2 <= radius**2] = 1.0
    elif kind == "gaussian":
        sigma = float(aperture["sigma_px"])
        radius = float(aperture.get("radius_px", 3.0 * sigma))
        mask = rr2 <= radius**2
        weights[mask] = np.exp(-0.5 * rr2[mask] / sigma**2)
    else:
        raise ValueError(f"Unknown aperture kind: {kind}")
    return weights


def same_radius_control_positions(
    object_yx,
    star_yx,
    ny,
    nx,
    n_positions=8,
    exclude_angle_deg=25.0,
    margin_px=4,
):
    """Return integer control positions at the same star-object radius."""

    oy, ox = map(float, object_yx)
    sy, sx = map(float, star_yx)
    dy = oy - sy
    dx = ox - sx
    radius = math.hypot(dy, dx)
    theta0 = math.atan2(dy, dx)

    controls = []
    for k in range(int(n_positions)):
        theta = theta0 + 2.0 * math.pi * k / float(n_positions)
        if angular_separation_deg(theta, theta0) < exclude_angle_deg:
            continue
        y = int(round(sy + radius * math.sin(theta)))
        x = int(round(sx + radius * math.cos(theta)))
        if margin_px <= y < ny - margin_px and margin_px <= x < nx - margin_px:
            controls.append((y, x))
    return controls


def _as_cube(cube_zyx, name="cube") -> np.ndarray:
    cube = np.asarray(cube_zyx, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected {name} with shape (nz,ny,nx), got {cube.shape}.")
    return cube


def annulus_background_spectrum(cube_zyx, center_yx, r_in, r_out, *, exclude_yx=None, exclude_radius=0.0):
    """Per-channel local background = median of a source-free annulus.

    Used for the wings-intact aperture-correction path: subtracting a distant
    annulus (rather than a local surface, stage04b) preserves the companion's
    PSF wings so the PSF growth-curve aperture correction stays self-consistent
    (box3<box5). Excludes a region around ``exclude_yx`` (the primary)."""

    cube = _as_cube(cube_zyx)
    _, ny, nx = cube.shape
    yy, xx = np.mgrid[0:ny, 0:nx]
    r = np.hypot(yy - float(center_yx[0]), xx - float(center_yx[1]))
    mask = (r >= float(r_in)) & (r <= float(r_out))
    if exclude_yx is not None and float(exclude_radius) > 0:
        mask &= np.hypot(yy - float(exclude_yx[0]), xx - float(exclude_yx[1])) > float(exclude_radius)
    if not mask.any():
        return np.zeros(cube.shape[0], dtype=np.float64)
    vals = cube[:, mask]
    with np.errstate(all="ignore"):
        return np.nanmedian(vals, axis=1).astype(np.float64)


def _flag_window(wave_A: np.ndarray, windows_A: Sequence[Sequence[float]], bit: int, flags: np.ndarray) -> None:
    for window in windows_A or ():
        if window is None or len(window) != 2:
            continue
        lo, hi = window
        if lo is None or hi is None:
            continue
        flags[(wave_A >= float(lo)) & (wave_A <= float(hi))] |= int(bit)


def channel_flags(
    wave_A,
    *,
    bad_windows_A: Sequence[Sequence[float]] = (),
    skyline_windows_A: Sequence[Sequence[float]] = (),
    interpolated_windows_A: Sequence[Sequence[float]] = (),
    clipped_mask=None,
    good_mask=None,
    bad_mask=None,
) -> np.ndarray:
    wave = np.asarray(wave_A, dtype=np.float64)
    flags = np.zeros(wave.size, dtype=np.int32)
    _flag_window(wave, bad_windows_A, FLAG_BAD_WINDOW, flags)
    _flag_window(wave, skyline_windows_A, FLAG_SKYLINE, flags)
    _flag_window(wave, interpolated_windows_A, FLAG_INTERPOLATED, flags)
    if good_mask is not None:
        flags[~np.asarray(good_mask, dtype=bool)] |= FLAG_BAD_WINDOW
    if bad_mask is not None:
        flags[np.asarray(bad_mask, dtype=bool)] |= FLAG_BAD_WINDOW
    if clipped_mask is not None:
        flags[np.asarray(clipped_mask, dtype=bool)] |= FLAG_CLIPPED
    return flags


def aperture_correction_from_psf(
    wave_A,
    aperture: dict,
    psf_model: dict | None,
    *,
    center_yx=(0.0, 0.0),
    correction_mode: str = "auto",
) -> tuple[np.ndarray, str, float]:
    """Return wavelength-dependent aperture correction from a C1 PSF model."""

    wave = np.asarray(wave_A, dtype=np.float64)
    mode = str(correction_mode or "auto").lower()
    if mode in {"none", "off", "false"}:
        return np.ones(wave.size, dtype=np.float64), "none", 0.0
    if psf_model is None:
        if mode in {"auto", "optional"}:
            return np.ones(wave.size, dtype=np.float64), "none", 0.0
        raise RuntimeError("Aperture correction requested but no psf_model was supplied.")

    norm_radius = float(psf_model.get("norm_radius_px", 25.0))
    half = int(math.ceil(norm_radius))
    frac_y = float(center_yx[0]) - round(float(center_yx[0]))
    frac_x = float(center_yx[1]) - round(float(center_yx[1]))
    source_center = (half + frac_y, half + frac_x)
    yy, xx = np.indices((2 * half + 1, 2 * half + 1), dtype=np.float64)
    dy = yy - source_center[0]
    dx = xx - source_center[1]
    weights = aperture_weights(2 * half + 1, 2 * half + 1, source_center, aperture)
    fractions = np.empty(wave.size, dtype=np.float64)
    for i, w in enumerate(wave):
        psf = evaluate_psf_model(psf_model, float(w), dy, dx)
        frac = float(np.nansum(psf * weights))
        if not np.isfinite(frac) or frac <= 0:
            raise RuntimeError(f"Invalid aperture PSF fraction at wave={w}.")
        fractions[i] = frac
    return (1.0 / fractions).astype(np.float64), "psf_growth_curve", norm_radius


def circular_window_indices(shape, center_yx, radius_px):
    ny, nx = map(int, shape)
    cy, cx = map(float, center_yx)
    radius = float(radius_px)
    half = int(math.ceil(radius))
    y1 = max(0, int(math.floor(cy)) - half)
    y2 = min(ny, int(math.floor(cy)) + half + 2)
    x1 = max(0, int(math.floor(cx)) - half)
    x2 = min(nx, int(math.floor(cx)) + half + 2)
    yy, xx = np.mgrid[y1:y2, x1:x2]
    mask = (yy - cy) ** 2 + (xx - cx) ** 2 <= radius**2
    return yy[mask].astype(int), xx[mask].astype(int)


def normalized_psf_window(psf_model, wavelength_A, center_yx, ypix, xpix):
    p = evaluate_psf_model(
        psf_model,
        float(wavelength_A),
        np.asarray(ypix, dtype=np.float64) - float(center_yx[0]),
        np.asarray(xpix, dtype=np.float64) - float(center_yx[1]),
    ).astype(np.float64)
    p[~np.isfinite(p)] = 0.0
    p[p < 0] = 0.0
    norm = float(np.sum(p))
    if not np.isfinite(norm) or norm <= 0:
        raise RuntimeError("PSF window has invalid normalization.")
    return p / norm


def covariance_factor_for_npix(npix_eff, covariance_factor_box3=1.0):
    """Linearly interpolate covariance inflation between one pixel and box3."""

    vals = np.asarray(npix_eff, dtype=np.float64)
    box3 = float(covariance_factor_box3)
    if not np.isfinite(box3) or box3 <= 0:
        box3 = 1.0
    t = np.clip((vals - 1.0) / 8.0, 0.0, 1.0)
    return 1.0 + t * (box3 - 1.0)


def _channel_estimate(data, variance, p, valid, *, clip_sigma=4.0, clip_max_iter=2):
    data = np.asarray(data, dtype=np.float64)
    variance = np.asarray(variance, dtype=np.float64)
    p = np.asarray(p, dtype=np.float64)
    valid = np.asarray(valid, dtype=bool)
    mask = valid.copy()
    n_valid = int(np.count_nonzero(valid))
    if n_valid == 0:
        return np.nan, np.nan, np.nan, 1.0, np.zeros_like(valid, dtype=bool)

    flux = np.nan
    raw_var = np.nan
    max_iter = max(0, int(clip_max_iter))
    for iteration in range(max_iter + 1):
        denom = np.sum((p[mask] ** 2) / variance[mask])
        if not np.isfinite(denom) or denom <= 0:
            return np.nan, np.nan, np.nan, 1.0, valid.copy()
        flux = float(np.sum(p[mask] * data[mask] / variance[mask]) / denom)
        raw_var = float(1.0 / denom)
        if clip_sigma is None or iteration >= max_iter:
            break
        z = np.zeros_like(data, dtype=np.float64)
        z[valid] = (data[valid] - flux * p[valid]) / np.sqrt(variance[valid])
        new_mask = valid & (np.abs(z) <= float(clip_sigma))
        if np.array_equal(new_mask, mask):
            break
        mask = new_mask

    clipped = valid & ~mask
    p_kept = p[mask]
    npix_eff = np.nan
    if p_kept.size and np.sum(p_kept**2) > 0:
        npix_eff = float((np.sum(p_kept) ** 2) / np.sum(p_kept**2))
    frac_clip = float(np.count_nonzero(clipped) / max(n_valid, 1))
    return flux, raw_var, npix_eff, frac_clip, clipped


def estimate_variance_cube(cube_zyx):
    cube = np.asarray(cube_zyx, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected cube shape (nz,ny,nx), got {cube.shape}.")
    out = np.empty_like(cube, dtype=np.float64)
    for i in range(cube.shape[0]):
        sigma = robust_sigma(cube[i])
        if not np.isfinite(sigma) or sigma <= 0:
            sigma = 1.0
        out[i] = sigma**2
    return out


def optimal_raw_spectrum(
    cube_zyx,
    variance_zyx,
    wave_A,
    center_yx,
    psf_model,
    *,
    window_radius_px=8.0,
    clip_sigma=4.0,
    clip_max_iter=2,
    n_jobs=1,
    bkg_spectrum=None,
):
    cube = np.asarray(cube_zyx, dtype=np.float64)
    variance = np.asarray(variance_zyx, dtype=np.float64)
    wave = np.asarray(wave_A, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected cube shape (nz,ny,nx), got {cube.shape}.")
    if variance.shape != cube.shape:
        raise ValueError("variance_zyx shape must match cube_zyx.")
    if wave.ndim != 1 or wave.size != cube.shape[0]:
        raise ValueError("wave_A must be 1D and match cube spectral length.")
    if bkg_spectrum is not None:
        bkg_spectrum = np.asarray(bkg_spectrum, dtype=np.float64)
        if bkg_spectrum.shape != wave.shape:
            raise ValueError("bkg_spectrum must match wave_A length.")

    nz, ny, nx = cube.shape
    ypix, xpix = circular_window_indices((ny, nx), center_yx, window_radius_px)
    flux = np.full(nz, np.nan, dtype=np.float64)
    raw_var = np.full(nz, np.nan, dtype=np.float64)
    npix_eff = np.full(nz, np.nan, dtype=np.float64)
    clip_fraction = np.zeros(nz, dtype=np.float64)
    chunk_rejection = {}

    def _estimate_range(z0, z1):
        # Per-channel work identical to the serial loop; the rejection counts
        # accumulate in a per-chunk map (integer-valued, so the final sum is
        # exact regardless of chunk order).
        local_map = np.zeros((ny, nx), dtype=np.float64)
        for z in range(z0, z1):
            data = cube[z, ypix, xpix]
            if bkg_spectrum is not None and np.isfinite(bkg_spectrum[z]):
                # Local background reference: subtracting a per-channel scalar
                # is separable from the Horne estimator (D1 v2 §3.1).
                data = data - bkg_spectrum[z]
            var = variance[z, ypix, xpix]
            p = normalized_psf_window(psf_model, wave[z], center_yx, ypix, xpix)
            valid = np.isfinite(data) & np.isfinite(var) & (var > 0) & np.isfinite(p) & (p > 0)
            f, v, neff, frac, clipped = _channel_estimate(
                data,
                var,
                p,
                valid,
                clip_sigma=clip_sigma,
                clip_max_iter=clip_max_iter,
            )
            flux[z] = f
            raw_var[z] = v
            npix_eff[z] = neff
            clip_fraction[z] = frac
            if np.any(clipped):
                np.add.at(local_map, (ypix[clipped], xpix[clipped]), 1.0)
        chunk_rejection[z0] = local_map

    run_channel_chunks(_estimate_range, nz, n_jobs=n_jobs)
    rejection_map = np.zeros((ny, nx), dtype=np.float64)
    for z0 in sorted(chunk_rejection):
        rejection_map += chunk_rejection[z0]

    return {
        "flux": flux,
        "variance": raw_var,
        "npix_eff": npix_eff,
        "clip_fraction": clip_fraction,
        "rejection_map": rejection_map,
    }


def control_optimal_spectra(
    cube_zyx,
    variance_zyx,
    wave_A,
    object_yx,
    star_yx,
    psf_model,
    *,
    window_radius_px=8.0,
    clip_sigma=4.0,
    clip_max_iter=2,
    n_controls=8,
    exclude_angle_deg=25.0,
    n_jobs=1,
    local_bkg_annulus_px=None,
):
    cube = np.asarray(cube_zyx, dtype=np.float64)
    _, ny, nx = cube.shape
    controls = same_radius_control_positions(
        object_yx,
        star_yx,
        ny,
        nx,
        n_positions=int(n_controls),
        exclude_angle_deg=float(exclude_angle_deg),
        margin_px=int(math.ceil(window_radius_px)) + 1,
    )
    spectra = []
    for center in controls:
        bkg = None
        if local_bkg_annulus_px is not None:
            bkg = annulus_background_spectrum(
                cube,
                center,
                local_bkg_annulus_px[0],
                local_bkg_annulus_px[1],
                exclude_yx=star_yx,
                exclude_radius=float(local_bkg_annulus_px[2]) if len(local_bkg_annulus_px) > 2 else 30.0,
            )
        raw = optimal_raw_spectrum(
            cube,
            variance_zyx,
            wave_A,
            center,
            psf_model,
            window_radius_px=window_radius_px,
            clip_sigma=clip_sigma,
            clip_max_iter=clip_max_iter,
            n_jobs=n_jobs,
            bkg_spectrum=bkg,
        )
        spectra.append(raw["flux"])
    if not spectra:
        return controls, np.empty((0, cube.shape[0]), dtype=np.float64)
    return controls, np.asarray(spectra, dtype=np.float64)


def psf_image(shape, wavelength_A, center_yx, psf_model):
    yy, xx = np.indices(shape, dtype=np.float64)
    return evaluate_psf_model(
        psf_model,
        float(wavelength_A),
        yy - float(center_yx[0]),
        xx - float(center_yx[1]),
    )


def fit_primary_psf_model_cube(
    cube_zyx,
    wave_A,
    primary_yx,
    psf_model,
    *,
    variance_zyx=None,
    fit_radius_px: float | None = None,
    exclude_centers_yx=(),
    exclude_radius_px: float = 8.0,
    n_jobs: int = 1,
) -> tuple[np.ndarray, dict]:
    cube = np.asarray(cube_zyx, dtype=np.float64)
    wave = np.asarray(wave_A, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected cube shape (nz,ny,nx), got {cube.shape}.")
    if wave.size != cube.shape[0]:
        raise ValueError("wave_A must match cube spectral length.")
    variance = None if variance_zyx is None else np.asarray(variance_zyx, dtype=np.float64)
    if variance is not None and variance.shape != cube.shape:
        raise ValueError("variance_zyx shape must match cube_zyx.")

    nz, ny, nx = cube.shape
    yy, xx = np.indices((ny, nx), dtype=np.float64)
    py, px = map(float, primary_yx)
    fit_radius = float(fit_radius_px or psf_model.get("norm_radius_px", 25.0))
    fit_mask = (yy - py) ** 2 + (xx - px) ** 2 <= fit_radius**2
    for center in exclude_centers_yx or ():
        if center is None:
            continue
        cy, cx = map(float, center)
        fit_mask &= (yy - cy) ** 2 + (xx - cx) ** 2 > float(exclude_radius_px) ** 2

    model = np.zeros_like(cube, dtype=np.float64)
    amplitudes = np.full(nz, np.nan, dtype=np.float64)
    backgrounds = np.full(nz, np.nan, dtype=np.float64)
    n_fit = np.zeros(nz, dtype=np.int32)

    def _fit_range(z0, z1):
        # Per-channel work identical to the serial loop; disjoint output slots.
        for z in range(z0, z1):
            psf = psf_image((ny, nx), wave[z], primary_yx, psf_model)
            data = cube[z]
            valid = fit_mask & np.isfinite(data) & np.isfinite(psf)
            if variance is not None:
                valid &= np.isfinite(variance[z]) & (variance[z] > 0)
                weight = 1.0 / variance[z][valid]
            else:
                weight = np.ones(np.count_nonzero(valid), dtype=np.float64)
            if np.count_nonzero(valid) < 3:
                continue
            a = np.column_stack([psf[valid], np.ones(np.count_nonzero(valid), dtype=np.float64)])
            sw = np.sqrt(weight)
            try:
                coeff, *_ = np.linalg.lstsq(a * sw[:, None], data[valid] * sw, rcond=None)
            except np.linalg.LinAlgError:
                continue
            amp = float(coeff[0])
            bg = float(coeff[1])
            amplitudes[z] = amp
            backgrounds[z] = bg
            n_fit[z] = int(np.count_nonzero(valid))
            model[z] = amp * psf

    run_channel_chunks(_fit_range, nz, n_jobs=n_jobs)
    meta = {
        "amplitude_median": None if not np.any(np.isfinite(amplitudes)) else float(np.nanmedian(amplitudes)),
        "background_median": None if not np.any(np.isfinite(backgrounds)) else float(np.nanmedian(backgrounds)),
        "n_fit_median": int(np.nanmedian(n_fit)) if n_fit.size else 0,
        "fit_radius_px": float(fit_radius),
        "exclude_radius_px": float(exclude_radius_px),
    }
    return model, meta


## 4 · Chequeo de deriva


In [ ]:
import ast as _ast, hashlib as _hashlib

_SHAS = {
    "musepipe/stats.py:finite_values": "8aa861655f2b",
    "musepipe/stats.py:robust_sigma": "ef2aa72a72de",
    "musepipe/stats.py:robust_sigma_axis0": "0b976272de28",
    "musepipe/apertures.py:angular_separation_deg": "ce3b83206746",
    "musepipe/apertures.py:aperture_weights": "d8e1fd8bb88d",
    "musepipe/apertures.py:same_radius_control_positions": "fb89fcf9dd7d",
    "musepipe/extraction/aperture.py:_as_cube": "67ede036d305",
    "musepipe/extraction/aperture.py:annulus_background_spectrum": "68ec4c4e7299",
    "musepipe/extraction/aperture.py:_flag_window": "7ba686789970",
    "musepipe/extraction/aperture.py:channel_flags": "dbceb28a0a0b",
    "musepipe/extraction/aperture.py:aperture_correction_from_psf": "8ed04307abec",
    "musepipe/extraction/optimal.py:circular_window_indices": "ef251183f52a",
    "musepipe/extraction/optimal.py:normalized_psf_window": "9e755174b9e5",
    "musepipe/extraction/optimal.py:covariance_factor_for_npix": "bfff5c5c63d8",
    "musepipe/extraction/optimal.py:_channel_estimate": "e8b851d36243",
    "musepipe/extraction/optimal.py:estimate_variance_cube": "52dd9aee1ded",
    "musepipe/extraction/optimal.py:optimal_raw_spectrum": "ddb22d1bcebd",
    "musepipe/extraction/optimal.py:control_optimal_spectra": "f99f750cdfa0",
    "musepipe/extraction/optimal.py:psf_image": "c646ec012650",
    "musepipe/extraction/optimal.py:fit_primary_psf_model_cube": "b012c1e0484a"
}

def chequeo_de_deriva(shas=_SHAS, root=ROOT):
    problemas = []
    for key, sha in shas.items():
        rel, name = key.rsplit(':', 1)
        text = (root / rel).read_text(encoding='utf-8')
        lines = text.splitlines(keepends=True)
        node = next((n for n in _ast.parse(text).body
                     if isinstance(n, _ast.FunctionDef) and n.name == name), None)
        if node is None:
            problemas.append(f'{key}: ya no existe en musepipe'); continue
        start = min([node.lineno] + [d.lineno for d in node.decorator_list]) - 1
        src = ''.join(lines[start:node.end_lineno]).rstrip('\n')
        actual = _hashlib.sha256(src.encode('utf-8')).hexdigest()[:12]
        if actual != sha:
            problemas.append(f'{key}: la copia es {sha}, musepipe tiene {actual}')
    return problemas

_deriva = chequeo_de_deriva()
if _deriva:
    print('DERIVA — la cadena cambió y esta copia se quedó atrás:')
    for p in _deriva:
        print('  ·', p)
    print(f'\nRegenera: python scripts/build_debug_notebooks.py --target {TARGET} C3')
else:
    print(f'sin deriva: las {len(_SHAS)} funciones copiadas son las de musepipe')


## 5 · El modelo de la primaria (lo que separa las dos variantes)

`psfsub` necesita restar la primaria antes de extraer. El ajuste es canal a canal, con la PSF de C1, **excluyendo un disco alrededor del compañero** para no absorberlo en el modelo de la estrella — si ese radio se queda corto, el modelo se come parte del compañero y `psfsub` sale bajo. Es una de las perillas interesantes de tocar.

*(Es la celda cara: ajusta un modelo por canal. Un par de minutos.)*


In [ ]:
primary_model, psfsub_meta = fit_primary_psf_model_cube(
    STAGE02, WAVE, STAR_YX, PSF_MODEL,
    variance_zyx=STAT_CUBE,
    fit_radius_px=PRIMARY_FIT_RADIUS,
    exclude_centers_yx=[OBJECT_YX],
    exclude_radius_px=PRIMARY_EXCL_RADIUS)
PSFSUB_CUBE = STAGE02 - primary_model
print('ajuste de la primaria:', {kk: psfsub_meta[kk] for kk in list(psfsub_meta)[:4]})

iz = int(np.argmin(np.abs(WAVE - 7500)))
y, x = int(round(STAR_YX[0])), int(round(STAR_YX[1]))
sl = (slice(y - 40, y + 41), slice(x - 40, x + 41))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, img, t in ((axes[0], STAGE02[iz][sl], 'B2 (con primaria)'),
                   (axes[1], primary_model[iz][sl], 'modelo de la primaria'),
                   (axes[2], PSFSUB_CUBE[iz][sl], 'residual = psfsub')):
    v = np.nanpercentile(np.abs(img), 99)
    ax.imshow(img, origin='lower', cmap='magma', vmin=-0.1 * v, vmax=v)
    ax.set_title(f'{t}  (λ={WAVE[iz]:.0f} Å)', fontsize=8)
fig.tight_layout(); plt.show()


## 6 · Las dos extracciones

El mismo estimador sobre los dos cubos. `optimal_raw_spectrum` devuelve además la varianza propagada, `npix_eff` y la fracción de píxeles rechazados por canal — el clipping es el que hay que vigilar: si se concentra en el compañero, se está recortando la señal (es el chequeo `v4_clip_concentration` de la spec).


In [ ]:
def extrae(cube, variance, etiqueta):
    # El peso del estimador es 1/varianza, y la varianza lleva el factor de M5.
    if variance is not None:
        variance = np.asarray(variance, dtype=float) * STAT_FACTOR
    else:
        variance = estimate_variance_cube(cube)
    bkg = None
    if LOCAL_BKG_ANNULUS_PX is not None:
        bkg = annulus_background_spectrum(
            cube, OBJECT_YX, LOCAL_BKG_ANNULUS_PX[0], LOCAL_BKG_ANNULUS_PX[1],
            exclude_yx=STAR_YX,
            exclude_radius=(LOCAL_BKG_ANNULUS_PX[2] if len(LOCAL_BKG_ANNULUS_PX) > 2 else 30.0))
    raw = optimal_raw_spectrum(cube, variance, WAVE, OBJECT_YX, PSF_MODEL,
                               window_radius_px=WINDOW_RADIUS_PX,
                               clip_sigma=CLIP_SIGMA, clip_max_iter=CLIP_MAX_ITER,
                               n_jobs=1, bkg_spectrum=bkg)
    cov = covariance_factor_for_npix(raw['npix_eff'], COV_FACTOR)
    raw_var = raw['variance'] * cov
    ctrl_yx, ctrl = control_optimal_spectra(
        cube, variance, WAVE, OBJECT_YX, STAR_YX, PSF_MODEL,
        window_radius_px=WINDOW_RADIUS_PX, clip_sigma=CLIP_SIGMA,
        clip_max_iter=CLIP_MAX_ITER, n_controls=N_CONTROLS,
        exclude_angle_deg=EXCLUDE_ANGLE_DEG, n_jobs=1,
        local_bkg_annulus_px=LOCAL_BKG_ANNULUS_PX)
    err_emp = (robust_sigma_axis0(ctrl) if ctrl.shape[0] >= 2
               else np.full(WAVE.size, robust_sigma(raw['flux'])))
    usable = (variance is not None and str(ERROR_MODE).lower() != 'empirical'
              and STAT_STATUS.lower() != 'red')
    err = np.sqrt(np.clip(raw_var, 0.0, np.inf)) if usable else np.asarray(err_emp, float)
    modo = 'stat' if usable else 'empirical'
    apert = {'kind': 'circle', 'radius_px': float(WINDOW_RADIUS_PX),
             'name': f'optimal_r{float(WINDOW_RADIUS_PX):g}'}
    apcorr, apcorr_mode, _nr = aperture_correction_from_psf(
        WAVE, apert, PSF_MODEL, center_yx=OBJECT_YX, correction_mode=APCORR_MODE)
    print(f'{etiqueta:8s} modo={modo:9s} apcorr={float(np.nanmedian(apcorr)):6.2f} '
          f'npix_eff={float(np.nanmedian(raw["npix_eff"])):6.1f} '
          f'clip_medio={100 * float(np.nanmedian(raw["clip_fraction"])):.2f}%')
    return {'raw': raw, 'flux': raw['flux'] * apcorr, 'err': err * apcorr,
            'err_emp': np.asarray(err_emp, float) * apcorr, 'apcorr': apcorr,
            'modo': modo, 'controles': ctrl, 'n_ctrl': len(ctrl_yx)}

LS     = extrae(LS_CUBE, STAT_CUBE, 'ls')
PSFSUB = extrae(PSFSUB_CUBE, STAT_CUBE, 'psfsub')


## 7 · Las dos variantes, una al lado de la otra

Es la comparación que D1 consume. Una diferencia **estructurada** entre ellas no es ruido: es el modelo de halo, porque el objeto y el estimador son los mismos y lo único que cambia es qué se restó antes.


In [ ]:
from musepipe.spectral import median_filter_1d
fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True,
                             gridspec_kw={'height_ratios': [2, 1]})
a1.plot(WAVE, median_filter_1d(LS['flux'], 41), lw=1.1, label='optimal_ls')
a1.plot(WAVE, median_filter_1d(PSFSUB['flux'], 41), lw=1.1, label='optimal_psfsub')
a1.axhline(0, color='0.7', lw=0.6); a1.axvline(6563, color='tab:red', ls=':')
a1.legend(fontsize=8); a1.set_ylabel('flujo (mediana 41 ch)')
a2.plot(WAVE, median_filter_1d(LS['flux'] - PSFSUB['flux'], 41), lw=1.0, color='tab:purple')
a2.axhline(0, color='0.7', lw=0.6)
a2.set_ylabel('ls − psfsub'); a2.set_xlabel('λ [Å]')
a1.set_title('las dos variantes: mismo estimador, distinto fondo', fontsize=9)
fig.tight_layout(); plt.show()
for nombre, v in (('ls', LS), ('psfsub', PSFSUB)):
    print(f"{nombre:8s} flujo mediano = {float(np.nanmedian(v['flux'])):9.2f}"
          f"  error mediano = {float(np.nanmedian(v['err'])):8.2f}  controles = {v['n_ctrl']}")


## 8 · Comparación con la cadena

Cada variante contra **su** producto. Con las perillas por defecto deben salir idénticas; si tocas `WINDOW_RADIUS_PX` o el radio de exclusión de la primaria, aquí se ve exactamente cuánto se movió cada una.


In [ ]:
from musepipe.extraction.product import SpectrumProduct

def compara(nombre, mio, fichero, rtol=1e-9):
    ref = SpectrumProduct.read(SD / fichero)
    ok = True
    print(f'{nombre} vs {fichero}:')
    for clave, a, b in (('flujo', mio['flux'], np.asarray(ref.flux, float)),
                        ('error', mio['err'], np.asarray(ref.flux_err, float)),
                        ('apcorr', mio['apcorr'], np.asarray(ref.apcorr, float))):
        fin = np.isfinite(a) & np.isfinite(b)
        d = np.abs(a - b)[fin]
        ig = np.isclose(a[fin], b[fin], rtol=rtol, atol=0.0)
        print(f'   {clave:7s} idénticos {100 * ig.mean():6.2f}% de {fin.sum()} canales'
              f' | máx |Δ| = {d.max():.3e}')
        ok &= bool(ig.all())
    return ok, ref

ok_ls, ref_ls = compara('optimal_ls    ', LS, 'spec_optimal_object.fits')
ok_ps, ref_ps = compara('optimal_psfsub', PSFSUB, 'spec_optimal_psfsub_object.fits')
print()
print('IDÉNTICO: la copia reproduce la cadena.' if (ok_ls and ok_ps) else
      'DIFIERE — si has tocado una perilla, es lo esperado; si no, revisa el chequeo de deriva.')

fig, axes = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
for ax, (nombre, mio, ref) in zip(axes, (('optimal_ls', LS, ref_ls),
                                         ('optimal_psfsub', PSFSUB, ref_ps))):
    ax.plot(WAVE, median_filter_1d(np.asarray(ref.flux, float), 41), lw=1.6,
            color='0.6', label='cadena')
    ax.plot(WAVE, median_filter_1d(mio['flux'], 41), lw=1.0, ls='--',
            color='tab:blue', label='este notebook')
    ax.set_ylabel(nombre, fontsize=9); ax.legend(fontsize=8)
axes[-1].set_xlabel('λ [Å]')
fig.tight_layout(); plt.show()
